# `phast` Quickstart Notebook

Five-minute interactive walk-through. Companion to `00_quickstart.md`.
Runs on CPU in ~2-5 minutes for the baseline-verification step count we use here.

## 1. Install

If you have not already done so, install in editable mode from the repo root:

```bash
pip install -e .
```

In [ ]:
import torch
import phast
print('phast:', phast.__file__)
print('torch:', torch.__version__, '| device available:',
      'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))

## 2. Load a YAML config

We use the COMSOL B7 dynamic-branching benchmark. Loading a config gives
you a `ProblemConfig` dataclass; `resolve_config` turns it into mesh,
material, BCs, and a `SolverConfig`.

In [ ]:
from phast.config import load_config
cfg = load_config('configs/benchmarks/dynamic/B7_dynamic_crack_branching_comsol.yaml')
print('Problem:', cfg.problem.name)
print('Material preset:', cfg.material.preset)
print('Energy split:', cfg.material.overrides.get('energy_split', '<from preset>'))
print('Solver:', cfg.solver.solver_type, '| damage_every:', cfg.solver.damage_every)

## 3. Override for a baseline verification

Cap the run at 50 explicit steps so the notebook finishes quickly. Real
B7 needs ~3000 steps to develop the Y-branch.

In [ ]:
cfg.loading.t_total = None       # disable auto step count from t_total
cfg.solver.num_steps = 50        # baseline-verification override
cfg.output.print_every = 10
cfg.output.fast = True


## 4. Resolve the config and inspect the mesh

In [ ]:
from phast.config import resolve_config
objs = if cfg.device is None:
    from phast.config import DeviceConfig
    cfg.device = DeviceConfig()
cfg.device.device = 'cpu'
objs = resolve_config(cfg)
mesh = objs['mesh']
mat  = objs['material']
print(f'mesh: {mesh.n_nodes} nodes, {mesh.n_elements} elements')
print(f'material: E={mat.E:.1f} MPa, nu={mat.nu}, Gc={mat.Gc:.4g} N/mm, l0={mat.l0} mm')
print(f'pf_model: {mat.pf_model}, energy_split: {mat.energy_split}')

## 5. Run the staggered solver

In [ ]:
from phast.staggered_solver import StaggeredSolver
solver = StaggeredSolver(mesh, mat, objs['bcs'], config=objs['solver_config'])
solver.f_ext = objs['bcs'].get_neumann_forces(mesh)
for step in range(cfg.solver.num_steps):
    solver.step_full()
    if step % 10 == 0:
        print(f'step {step:4d}: max(d) = {solver.d.max().item():.4f}')
print('done. final max(d) =', solver.d.max().item())

## 6. Plot the damage field

At 50 steps the crack will not have branched yet; you should see a thin
diffuse damage band near the pre-notch tip. Run the full config from the
CLI to see the Y-branch develop.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
x = mesh.nodes[:, 0].cpu().numpy()
y = mesh.nodes[:, 1].cpu().numpy()
tri = mtri.Triangulation(x, y, mesh.elements.cpu().numpy())
fig, ax = plt.subplots(figsize=(8, 4))
tcf = ax.tricontourf(tri, solver.d.cpu().numpy(), levels=20, cmap='inferno')
fig.colorbar(tcf, ax=ax, label='damage d')
ax.set_aspect('equal')
ax.set_title(f'damage at step {cfg.solver.num_steps}')
plt.show()

## 7. Next steps

- Run the full benchmark from the CLI:

  ```bash
  python -m phast run configs/benchmarks/dynamic/B7_dynamic_crack_branching_comsol.yaml --device cpu
  ```

- Read the [phase-field primer](01_phase_field_primer.md).
- Build your own config: [setting up your problem](03_setting_up_your_problem.md).
- Try the suggested experiments: [exploration experiments](04_exploration_experiments.md).